In [6]:
import pandas as pd
import sqlite3

base = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/"
tables = ["categories", "customers", "employees", "orders", "order_details", "products", "shippers", "suppliers"]

conn = sqlite3.connect("northwind.db")

for t in tables:
    df = pd.read_csv("/content/northwind_dataset_task25.csv")
    if t == "orders":
        for col in ["orderDate", "requiredDate", "shippedDate"]:
            df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")
    df.to_sql(t, conn, index=False, if_exists="replace")
    print(t, df.shape)

def q(sql):
    return pd.read_sql_query(sql, conn)

categories (2155, 29)
customers (2155, 29)
employees (2155, 29)
orders (2155, 29)
order_details (2155, 29)
products (2155, 29)
shippers (2155, 29)
suppliers (2155, 29)


In [7]:
q("""
SELECT
  (SELECT COUNT(*) FROM order_details) AS detail_rows,
  (SELECT COUNT(*) FROM order_details od JOIN orders o ON od.orderID = o.orderID) AS after_orders_join,
  (SELECT COUNT(*) FROM order_details od
     JOIN orders o ON od.orderID = o.orderID
     JOIN products p ON od.productID = p.productID
     JOIN customers c ON o.customerID = c.customerID) AS after_all_joins
""")

,detail_rows,after_orders_join,after_all_joins
0,2155,7059,10850563


In [8]:
q("""
SELECT
  COUNT(DISTINCT o.orderID) AS total_orders,
  COUNT(DISTINCT o.customerID) AS total_customers,
  ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS total_revenue,
  ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)) / COUNT(DISTINCT o.orderID), 2) AS avg_order_value
FROM orders o
JOIN order_details od ON o.orderID = od.orderID
""")

,total_orders,total_customers,total_revenue,avg_order_value
0,830,89,3996238.19,4814.74


In [10]:
q("SELECT * FROM customers LIMIT 3")

,orderID,productID,unitPrice,quantity,discount,customerID,employeeID,orderDate,requiredDate,shippedDate,...,categoryID,quantityPerUnit,discontinued,categoryName,supplierCompany,supplierCountry,employeeName,employeeTitle,shipperName,lineTotal
0,10248,11,14.0,12,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,...,4,1 kg pkg.,0,Dairy Products,Cooperativa de Quesos 'Las Cabras',Spain,Steven Buchanan,Sales Manager,Federal Shipping,168.0
1,10248,42,9.8,10,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,...,5,32 - 1 kg pkgs.,1,Grains/Cereals,Leka Trading,Singapore,Steven Buchanan,Sales Manager,Federal Shipping,98.0
2,10248,72,34.8,5,0.0,VINET,5,1996-07-04,1996-08-01,1996-07-16,...,4,24 - 200 g pkgs.,0,Dairy Products,Formaggi Fortini s.r.l.,Italy,Steven Buchanan,Sales Manager,Federal Shipping,174.0


In [11]:
df = pd.read_csv("/content/northwind_dataset_task25.csv")
df.to_sql("sales", conn, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql_query(sql, conn)

In [12]:
q("""
SELECT customerCountry,
       COUNT(DISTINCT orderID) AS orders,
       ROUND(SUM(lineTotal), 2) AS revenue
FROM sales
GROUP BY customerCountry
ORDER BY revenue DESC
LIMIT 5
""")

,customerCountry,orders,revenue
0,USA,122,245584.62
1,Germany,122,230284.63
2,Austria,40,128003.85
3,Brazil,83,106925.78
4,France,77,81358.32


In [13]:
q("""
SELECT cat.categoryName,
       SUM(od.quantity) AS units_sold,
       ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue
FROM order_details od
JOIN products p ON od.productID = p.productID
JOIN categories cat ON p.categoryID = cat.categoryID
GROUP BY cat.categoryName
ORDER BY revenue DESC
""")

,categoryName,units_sold,revenue
0,Dairy Products,142693518,3.762374e+09
1,Beverages,144156492,3.191318e+09
2,Confections,86076810,1.801735e+09
3,Seafood,81483600,1.340234e+09
4,Meat/Poultry,24076756,8.961889e+08
5,Grains/Cereals,28853944,6.874426e+08
6,Condiments,25364232,4.772591e+08
7,Produce,12314800,4.522919e+08


In [14]:
q("""
SELECT p.productName,
       ROUND(SUM(od.unitPrice * od.quantity * (1 - od.discount)), 2) AS revenue
FROM order_details od
JOIN products p ON od.productID = p.productID
GROUP BY p.productName
ORDER BY revenue DESC
LIMIT 10
""")

,productName,revenue
0,Raclette Courdavault,3842407.80
1,Côte de Blaye,3393521.64
2,Thüringer Rostbratwurst,2571797.50
3,Camembert Pierrot,2388099.48
4,Tarte au sucre,2267278.56
5,Gnocchi di nonna Alice,2129653.00
6,Manjimup Dried Apples,1630966.35
7,Alice Mutton,1209840.06
8,Mozzarella di Giovanni,946204.94
9,Rössle Sauerkraut,847989.12


In [5]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("northwind.db")

df = pd.read_csv("/content/northwind_dataset_task25.csv")
df.to_sql("sales", conn, index=False, if_exists="replace")

def q(sql):
    return pd.read_sql_query(sql, conn)

In [6]:
q("SELECT COUNT(*) AS rows, COUNT(DISTINCT orderID) AS orders, ROUND(SUM(lineTotal), 2) AS revenue FROM sales")

,rows,orders,revenue
0,2155,830,1265793.02


In [8]:
q("""
SELECT employeeName,
       COUNT(DISTINCT orderID) AS orders,
       ROUND(SUM(lineTotal), 2) AS revenue
FROM sales
GROUP BY employeeName
ORDER BY revenue DESC
""")

,employeeName,orders,revenue
0,Margaret Peacock,156,232890.87
1,Janet Leverling,127,202812.82
2,Nancy Davolio,123,192107.59
3,Andrew Fuller,96,166537.76
4,Laura Callahan,104,126862.27
5,Robert King,72,124568.22
6,Anne Dodsworth,43,77308.06
7,Michael Suyama,67,73913.14
8,Steven Buchanan,42,68792.29


In [9]:
q("""
SELECT strftime('%Y-%m', orderDate) AS month,
       ROUND(SUM(lineTotal), 2) AS revenue
FROM sales
GROUP BY month
ORDER BY month
""")

,month,revenue
0,1996-07,27861.89
1,1996-08,25485.28
2,1996-09,26381.40
3,1996-10,37515.73
4,1996-11,45600.04
5,1996-12,45239.63
6,1997-01,61258.08
7,1997-02,38483.64
8,1997-03,38547.21
9,1997-04,53032.95
